In [2]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# 1. Setup API Key (Replace with your actual key or set in env)
os.environ["OPENAI_API_KEY"] = "sk-..."

# 2. Prepare Data (Augmentation Source)
# Creating a dummy document for demonstration
raw_text = "LangChain as_retriever converts a vector store into a retriever interface. " \
           "It is used for the augmentation phase in RAG to fetch relevant context."
loader = TextLoader.from_texts([raw_text])
documents = loader.load()

# Split text into chunks
text_splitter = CharacterTextSplitter(chunk_size=50, chunk_overlap=0)
docs = text_splitter.split_documents(documents)

# 3. Create Vector Store & Retriever (The Augmentation Step)
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embeddings)

# This is the specific method you asked about:
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

# 4. Setup Generation (The LLM Step)
llm = ChatOpenAI(model="gpt-3.5-turbo")

# Define how the context and question are combined
prompt_template = """Answer the question based only on the following context:
Context: {context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(prompt_template)

# 5. Build the Chain (Connecting Augmentation + Generation)
# Format: {question} -> Retriever (Augmentation) -> Context -> Prompt -> LLM (Generation)
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
)

# 6. Execute
query = "What does as_retriever do?"
response = rag_chain.invoke(query)

print(f"Question: {query}")
print(f"Answer: {response.content}")